In [1]:
import pandas as pd

df_anime = pd.read_csv('anime.csv')
df_ratings = pd.read_parquet('ratings.parquet')

In [3]:
df_ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6144926 entries, 0 to 6144925
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   user_id   int32
 1   anime_id  int16
 2   rating    Int8 
dtypes: Int8(1), int16(1), int32(1)
memory usage: 46.9 MB


In [3]:
import urllib.request

import math
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [13]:
df_ratings

,user_id,anime_id,rating
0,0,0,8
1,0,1,6
2,0,2,9
3,0,3,10
4,0,4,9
...,...,...,...
6144921,47142,352,8
6144922,47142,51,7
6144923,47142,54,7
6144924,47142,1784,9


In [10]:
NUM_USERS = df_ratings['user_id'].nunique(dropna=True)
NUM_ITEMS = df_ratings['anime_id'].nunique(dropna=True)
print(f'Hay {NUM_USERS} usuarios y {NUM_ITEMS} animes')
latent_dim = 5
epochs = 10

Hay 47143 usuarios y 6532 animes


In [24]:
print(df_ratings['rating'].min())
print(df_ratings['user_id'].min())
print(df_ratings['anime_id'].min())

1
0
0


In [11]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

class GMFModel(nn.Module):
    def __init__(self, num_users, num_items, latent_dim):
        super().__init__()
        # Definimos las capas de embedding para usuarios e ítems
        # Sumamos 1 al rango por si los IDs empiezan en 1 o hay índices máximos
        self.user_embedding = nn.Embedding(num_users + 1, latent_dim)
        self.item_embedding = nn.Embedding(num_items + 1, latent_dim)
        # Capa de salida para proyectar el producto escalar a una sola neurona (voto)
        self.fc = nn.Linear(latent_dim, 1)

    def forward(self, user_ids, item_ids):
        user_vec = self.user_embedding(user_ids)
        item_vec = self.item_embedding(item_ids)

        # Combinamos mediante producto elemento a elemento (equivalente a dot product)
        interact = torch.mul(user_vec, item_vec)

        # Salida escalar (voto estimado)
        return self.fc(interact).flatten()

# Instanciamos el modelo
GMF = GMFModel(NUM_USERS, NUM_ITEMS, latent_dim)
print(GMF)

GMFModel(
  (user_embedding): Embedding(47144, 5)
  (item_embedding): Embedding(6533, 5)
  (fc): Linear(in_features=5, out_features=1, bias=True)
)


In [18]:
# Configuración de optimizador y pérdida
optimizer = torch.optim.Adam(GMF.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

# Preparación de datos
users_t = torch.tensor(X_train[0], dtype=torch.long)
items_t = torch.tensor(X_train[1], dtype=torch.long)
ratings_t = torch.tensor(y_train, dtype=torch.float32)
dataset = TensorDataset(users_t, items_t, ratings_t)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

# Bucle de entrenamiento
for epoch in range(epochs):
    GMF.train()
    total_loss = 0
    for batch_u, batch_i, batch_r in loader:
        optimizer.zero_grad()
        # Predicción
        outputs = GMF(batch_u, batch_i)
        # Cálculo de error
        loss = loss_fn(outputs, batch_r)
        # Backpropagation
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Época {epoch+1}/{epochs} - Pérdida (MSE): {total_loss/len(loader):.4f}")

NameError: name 'X_train' is not defined

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Mapeo de IDs a índices (Label Encoding) para evitar errores en los Embeddings
df_ratings['user_idx'] = df_ratings['user_id'].astype('category').cat.codes
df_ratings['item_idx'] = df_ratings['anime_id'].astype('category').cat.codes

# Guardamos los conteos para las dimensiones de los modelos
NUM_USERS = df_ratings['user_idx'].nunique()
NUM_ITEMS = df_ratings['item_idx'].nunique()

# 2. División en entrenamiento y test
X = df_ratings[['user_idx', 'item_idx']].values
y = df_ratings['rating'].values

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Separamos los arrays para que el modelo los reciba bien
X_train = [X_train_val[:, 0], X_train_val[:, 1]]
y_train = y_train_val

# Parámetros básicos
latent_dim = 16
epochs = 5

# 2.7 s

In [20]:
# --- MODELO GMF ---
class GMFModel(nn.Module):
    def __init__(self, num_users, num_items, latent_dim):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, latent_dim)
        self.item_embedding = nn.Embedding(num_items, latent_dim)
        self.fc = nn.Linear(latent_dim, 1)

    def forward(self, user_ids, item_ids):
        u_emb = self.user_embedding(user_ids)
        i_emb = self.item_embedding(item_ids)
        interact = torch.mul(u_emb, i_emb)
        return self.fc(interact).flatten()

# --- MODELO MLP ---
class MLPModel(nn.Module):
    def __init__(self, num_users, num_items, latent_dim):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, latent_dim)
        self.item_embedding = nn.Embedding(num_items, latent_dim)
        self.mlp = nn.Sequential(
            nn.Linear(latent_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, user_ids, item_ids):
        u_emb = self.user_embedding(user_ids)
        i_emb = self.item_embedding(item_ids)
        vector = torch.cat([u_emb, i_emb], dim=-1)
        return self.mlp(vector).flatten()

In [ ]:
# Inicializar modelo, pérdida y optimizador
model = MLPModel(NUM_USERS, NUM_ITEMS, latent_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

# Preparar DataLoader
users_t = torch.tensor(X_train[0], dtype=torch.long)
items_t = torch.tensor(X_train[1], dtype=torch.long)
ratings_t = torch.tensor(y_train, dtype=torch.float32)

dataset = TensorDataset(users_t, items_t, ratings_t)
loader = DataLoader(dataset, batch_size=512, shuffle=True)

# Bucle de entrenamiento
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for b_u, b_i, b_r in loader:
        optimizer.zero_grad()
        preds = model(b_u, b_i)
        loss = loss_fn(preds, b_r)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Época {epoch+1}: Pérdida {total_loss/len(loader):.4f}")

# Evaluación final en Test
from sklearn.metrics import mean_absolute_error
model.eval()
with torch.no_grad():
    u_test = torch.tensor(X_test[:, 0], dtype=torch.long)
    i_test = torch.tensor(X_test[:, 1], dtype=torch.long)
    y_pred = model(u_test, i_test).numpy()

print(f"MAE en Test: {mean_absolute_error(y_test, y_pred):.4f}")
# 29 min

Época 1: Pérdida 2.1498
Época 2: Pérdida 1.4789
Época 3: Pérdida 1.4372
Época 4: Pérdida 1.4212
Época 5: Pérdida 1.4070
MAE en Test: 0.9175


In [25]:
torch.save(GMF.state_dict(), 'gmf_model_weights.pth')
print("Pesos del modelo guardados correctamente.")

Pesos del modelo guardados correctamente.


In [ ]:
# 1. Pasamos el tamaño final directamente (47144 y 6533)
# ya que tu clase GMFModel actual NO suma el +1 internamente.
modelo_cargado = GMFModel(num_users=47144, num_items=6533, latent_dim=5)

# 2. Cargamos los pesos del archivo
modelo_cargado.load_state_dict(torch.load('gmf_model_weights.pth'))

# 3. Modo evaluación
modelo_cargado.eval()
print("¡Modelo cargado con éxito!")

¡Modelo cargado con éxito!


In [34]:
from sklearn.metrics import mean_absolute_error

# 1. Aseguramos que el modelo esté en modo evaluación
modelo_cargado.eval()

# 2. Realizamos las predicciones sobre el conjunto de test
# Nota: X_test[:, 0] son los índices de usuarios y X_test[:, 1] los de animes
with torch.no_grad():
    # Convertimos los arrays de test a tensores de PyTorch
    u_test_t = torch.tensor(X_test[:, 0], dtype=torch.long)
    i_test_t = torch.tensor(X_test[:, 1], dtype=torch.long)

    # Obtenemos las predicciones del modelo cargado
    y_pred = modelo_cargado(u_test_t, i_test_t).numpy()

# 3. Calculamos el error comparando con las notas reales (y_test)
mae_final = mean_absolute_error(y_test, y_pred)

print(f"El MAE del modelo cargado es: {mae_final:.4f}")

El MAE del modelo cargado es: 8.0943


In [30]:
import torch

# Probamos con un usuario y un anime (usando los índices mapeados)
user_idx = torch.tensor([47142], dtype=torch.long)
anime_idx = torch.tensor([778], dtype=torch.long)

modelo_cargado.eval()
with torch.no_grad():
    prediccion = modelo_cargado(user_idx, anime_idx)
    print(f"La nota estimada es: {prediccion.item():.2f}")

La nota estimada es: -1.05
